In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import scipy
import os
import json

## Get the vector field

In [2]:
# direction of the current
def get_angle(u, v):
    return np.arctan(v/(u+1e-8))

def get_magnitude(u, v):
    return np.hypot(u, v)

def get_gradient(u, ax):
    return np.gradient(u, axis = ax)

def get_divergence(u, v):
    return get_gradient(u, 0) + get_gradient(v, 1)

def get_curl(u, v):
    return get_gradient(v, 0) - get_gradient(u, 1)

In [4]:
step = 1
angles, divs, mags, curls, uarr, varr = [],[],[],[],[],[]
for i in range(1,len(os.listdir("./vectors/vectors"))+1,step):
    data = np.load(f"vectors/vectors/frame_{i:05}.npz")
    
    u = data["u"]
    v = data["v"]

    # u = cv2.resize(u, None, fx = 0.5, fy = 0.5, interpolation=cv2.INTER_LINEAR)
    # v = cv2.resize(v, None, fx = 0.5, fy = 0.5, interpolation=cv2.INTER_LINEAR)
    
    # Y, X = np.mgrid[
    #     0:u.shape[0]:step,
    #     0:u.shape[1]:step
    # ]
    # angle_map = get_angle(u,v)
    # divergence_map = get_divergence(u,v)
    # magnitude_map = get_magnitude(u, v)
    # curl_map = get_curl(u,v)

    # angles.append(angle_map)
    # divs.append(divergence_map)
    # mags.append(magnitude_map)
    # curls.append(curl_map)
    uarr.append(u)
    varr.append(v)

# plt.figure(figsize=(12, 8))

# plt.quiver(
#     X,
#     Y,
#     u[::step, ::step],
#     # You might notice the minus sign, this is because for image coordinates, there is an increasing Y downwards but for matplotlib, it assumes Y increases upwards, small issue to take note of
#     -v[::step, ::step]
# )

# plt.gca().invert_yaxis()
# plt.show()

In [41]:
def get_characteristic_frequencies(field_3d, dt, top_k=5):
    """Applies a 3D window and rfftn, returning top temporal frequencies."""
    Nt, Ny, Nx = field_3d.shape[0], field_3d.shape[1], field_3d.shape[2]
    
    # Apply a 3D Hann window to suppress spectral leakage from video boundaries
    win_t = np.hanning(Nt)[:, None, None]
    win_y = np.hanning(Ny)[None, :, None]
    win_x = np.hanning(Nx)[None, None, :]
    windowed_field = field_3d * (win_t * win_y * win_x)
    
    # Compute 3D Real FFT across (time, y, x)
    spectrum = np.abs(scipy.fft.rfftn(windowed_field, workers=-1))
    
    # Marginalize (sum energy) over spatial frequencies (ky, kx) to get temporal profile
    temporal_energy = np.sum(spectrum, axis=(1, 2))
    freqs = scipy.fft.rfftfreq(Nt, d=dt)
    
    # Ignore 0 Hz (DC offset / static background flow)
    temporal_energy[0] = 0
    
    # Find the top K dominant frequencies
    top_indices = np.argsort(temporal_energy)[::-1][:top_k]
    return freqs[top_indices], temporal_energy[top_indices]

fps = 1/30
dt = fps * step
div_freqs, div_amps = get_characteristic_frequencies(np.array(divs), dt)
curl_freqs, curl_amps = get_characteristic_frequencies(np.array(curls), dt)
mag_freqs, mag_amps = get_characteristic_frequencies(np.array(mags), dt)

IndexError: index 102 is out of bounds for axis 0 with size 52

In [8]:
# 3. Musical Scale Mapping (C Major Pentatonic across 3 octaves: C3 to A5)
# Frequencies in Hz
PENTATONIC_HZ = np.array([
    130.81, 146.83, 164.81, 196.00, 220.00,  # Octave 3 (Bass / Divergence)
    261.63, 293.66, 329.63, 392.00, 440.00,  # Octave 4 (Mid / Curl)
    523.25, 587.33, 659.25, 783.99, 880.00   # Octave 5 (High)
])

def freq_to_musical_hz(raw_freq, max_nyquist=15.0, octave_offset=0):
    """Maps a 0–15 Hz ocean oscillation frequency to a pleasant musical note."""
    # Normalize log-scaled frequency to [0, 1]
    norm = np.clip(np.log1p(raw_freq) / np.log1p(max_nyquist), 0, 0.999)
    idx = int(norm * 5) + (octave_offset * 5)
    return float(PENTATONIC_HZ[np.clip(idx, 0, len(PENTATONIC_HZ) - 1)])


In [ ]:
#low memory code
win_t = 60          # 2.0-second temporal window - chunksize
hop_t = 15          # Update musical grid every 0.5 seconds (136 time steps total)
patch_xy = 25       # 16x16 spatial neighborhood per grid cell
fps = 30.0
dt = 1.0 / fps

num_files = len(os.listdir("./vectors/vectors"))
chunks = (num_files - win_t)/hop_t + 1
time_steps = range(0, Nt - win_t, hop_t)

data = np.load(f"vectors/vectors/frame_{1:05}.npz")
ys, xs = data["u"].shape[0], data["v"].shape[1] 

freqs = scipy.fft.rfftfreq(win_t, d=dt)

for t_idx, t_start in enumerate(time_steps):
    #extract the first 60, then hop 15 each time. so there (N - 60)/15 + 1
    uarr, varr = np.zeros(win_t, ys, xs)
    if t_idx > 0:
        uarr = uarr[win_t:, :, :]
        varr = varr[win_t:, :, :]

        max_win = hop_t
    else:
        max_win = win_t

    uls, vls = [], []
    for j in range(max_win):
        idx = t_start + j
        data = np.load(f"vectors/vectors/frame_{idx+1:05}.npz")
            
        u = data["u"]
        v = data["v"]

        uls.append(u)
        vls.append(v)
        
    u_add_arr, v_add_arr = np.array(uls), np.array(vls)
    uarr = np.concatenate([uarr, u_add_arr], axis = 0])
    varr = np.concatenate([varr, v_add_arr], axis = 0])
    #uarr, varr = np.array(uarr), np.array(varr)

    du_dy, du_dx = np.gradient(uarr, axis=(1, 2))
    dv_dy, dv_dx = np.gradient(varr, axis=(1, 2))

    divergence = du_dx + dv_dy
    curl = dv_dx - du_dy
    magnitude = np.hypot(uarr, varr)
    
    grid_y = Ny // patch_xy
    grid_x = Nx // patch_xy
    
    # Output array: [time_step, grid_y, grid_x, (curl_hz, div_hz, volume)]
    sonification_grid = np.zeros((len(time_steps), grid_y, grid_x, 3), dtype=np.float32)
    
    hanning_3d = (np.hanning(win_t)[:, None, None] * np.hanning(patch_xy)[None, :, None] * np.hanning(patch_xy)[None, None, :])

    
    for gy in range(grid_y):
        for gx in range(grid_x):
            y0, y1 = gy * patch_xy, (gy + 1) * patch_xy
            x0, x1 = gx * patch_xy, (gx + 1) * patch_xy
            
            # Extract local 3D cubes
            c_cube = curl[t_start:t_start+win_t, y0:y1, x0:x1] * hanning_3d
            d_cube = divergence[t_start:t_start+win_t, y0:y1, x0:x1] * hanning_3d
            m_cube = magnitude[t_start:t_start+win_t, y0:y1, x0:x1]
            
            # 3D FFT -> sum spatial wavenumbers to get dominant temporal oscillation
            c_spec = np.sum(np.abs(scipy.fft.rfftn(c_cube)), axis=(1, 2))
            d_spec = np.sum(np.abs(scipy.fft.rfftn(d_cube)), axis=(1, 2))
            c_spec = d_spec = 0  # Ignore 0 Hz static offset
            
            dom_curl_freq = freqs[np.argmax(c_spec)]
            dom_div_freq = freqs[np.argmax(d_spec)]
            
            # Map Curl -> Mid-range note, Divergence -> Bass note, Magnitude -> Volume
            sonification_grid[t_idx, gy, gx, 0] = freq_to_musical_hz(dom_curl_freq, octave_offset=1)
            sonification_grid[t_idx, gy, gx, 1] = freq_to_musical_hz(dom_div_freq, octave_offset=0)
            sonification_grid[t_idx, gy, gx, 2] = float(np.mean(m_cube))
    
    # Normalize volume to [0.0, 1.0]
    max_mag = np.max(sonification_grid[..., 2])
    if max_mag > 0:
        sonification_grid[..., 2] /= max_mag
        

In [5]:
# 1. Load your precomputed ocean current arrays: shape (2040, Ny, Nx)
# Replace these dummy arrays with np.load('u.npy') and np.load('v.npy')
uarr = np.array(uarr)
varr = np.array(varr)
Nt, Ny, Nx = uarr.shape[0], uarr.shape[1], uarr.shape[2]
fps = 30.0
dt = 1.0 / fps
# u = np.random.randn(Nt, Ny, Nx).astype(np.float32)
# v = np.random.randn(Nt, Ny, Nx).astype(np.float32)

# 2. Compute Divergence, Curl, and Magnitude
du_dy, du_dx = np.gradient(uarr, axis=(1, 2))
dv_dy, dv_dx = np.gradient(varr, axis=(1, 2))

divergence = du_dx + dv_dy
curl = dv_dx - du_dy
magnitude = np.hypot(uarr, varr)


# 4. Sliding 3D Spatiotemporal FFT over space-time blocks
win_t = 60          # 2.0-second temporal window
hop_t = 15          # Update musical grid every 0.5 seconds (136 time steps total)
patch_xy = 16       # 16x16 spatial neighborhood per grid cell

time_steps = range(0, Nt - win_t, hop_t)
grid_y = Ny // patch_xy
grid_x = Nx // patch_xy

# Output array: [time_step, grid_y, grid_x, (curl_hz, div_hz, volume)]
sonification_grid = np.zeros((len(time_steps), grid_y, grid_x, 3), dtype=np.float32)
freqs = scipy.fft.rfftfreq(win_t, d=dt)
hanning_3d = (np.hanning(win_t)[:, None, None] * np.hanning(patch_xy)[None, :, None] * np.hanning(patch_xy)[None, None, :])

for t_idx, t_start in enumerate(time_steps):
    for gy in range(grid_y):
        for gx in range(grid_x):
            y0, y1 = gy * patch_xy, (gy + 1) * patch_xy
            x0, x1 = gx * patch_xy, (gx + 1) * patch_xy
            
            # Extract local 3D cubes
            c_cube = curl[t_start:t_start+win_t, y0:y1, x0:x1] * hanning_3d
            d_cube = divergence[t_start:t_start+win_t, y0:y1, x0:x1] * hanning_3d
            m_cube = magnitude[t_start:t_start+win_t, y0:y1, x0:x1]
            
            # 3D FFT -> sum spatial wavenumbers to get dominant temporal oscillation
            c_spec = np.sum(np.abs(scipy.fft.rfftn(c_cube)), axis=(1, 2))
            d_spec = np.sum(np.abs(scipy.fft.rfftn(d_cube)), axis=(1, 2))
            c_spec = d_spec = 0  # Ignore 0 Hz static offset
            
            dom_curl_freq = freqs[np.argmax(c_spec)]
            dom_div_freq = freqs[np.argmax(d_spec)]
            
            # Map Curl -> Mid-range note, Divergence -> Bass note, Magnitude -> Volume
            sonification_grid[t_idx, gy, gx, 0] = freq_to_musical_hz(dom_curl_freq, octave_offset=1)
            sonification_grid[t_idx, gy, gx, 1] = freq_to_musical_hz(dom_div_freq, octave_offset=0)
            sonification_grid[t_idx, gy, gx, 2] = float(np.mean(m_cube))

# Normalize volume to [0.0, 1.0]
max_mag = np.max(sonification_grid[..., 2])
if max_mag > 0:
    sonification_grid[..., 2] /= max_mag

# Export lightweight JSON for the browser player
with open("ocean_sonification.json", "w") as f:
    json.dump({
        "hop_seconds": hop_t / fps,
        "grid_y": grid_y,
        "grid_x": grid_x,
        "data": np.round(sonification_grid, 2).tolist()
    }, f)

print("Saved ocean_sonification.json successfully.")


Saved ocean_sonification.json successfully.


In [7]:
sonification_grid

array([[[[2.61630005e+02, 1.30809998e+02, 5.94990030e-02],
         [2.61630005e+02, 1.30809998e+02, 4.64895368e-02],
         [2.61630005e+02, 1.30809998e+02, 8.33207741e-02],
         ...,
         [2.61630005e+02, 1.30809998e+02, 3.05795670e-02],
         [2.61630005e+02, 1.30809998e+02, 2.85867248e-02],
         [2.61630005e+02, 1.30809998e+02, 5.20055182e-02]],

        [[2.61630005e+02, 1.30809998e+02, 2.84563303e-02],
         [2.61630005e+02, 1.30809998e+02, 5.59977330e-02],
         [2.61630005e+02, 1.30809998e+02, 1.18670128e-01],
         ...,
         [2.61630005e+02, 1.30809998e+02, 3.73143032e-02],
         [2.61630005e+02, 1.30809998e+02, 4.05909047e-02],
         [2.61630005e+02, 1.30809998e+02, 3.70592438e-02]],

        [[2.61630005e+02, 1.30809998e+02, 7.02604875e-02],
         [2.61630005e+02, 1.30809998e+02, 9.16409194e-02],
         [2.61630005e+02, 1.30809998e+02, 1.04008019e-01],
         ...,
         [2.61630005e+02, 1.30809998e+02, 5.05931862e-02],
         [